# 05: Interpretation & Explainability

This notebook demonstrates the explainability engine and product affinity communities, confirming that recommendations make business sense.

In [ ]:
import pandas as pd
import sys
import os
import networkx as nx
import matplotlib.pyplot as plt
sys.path.append(os.path.abspath(".."))

from src.db import query_to_dataframe
from src.recommenders import HybridRecommender, PopularityRecommender, ItemCFRecommender, AssociationRuleRecommender
from src.interpretation import ExplainabilityEngine, build_product_affinity_network, get_communities

## 1. Load Data & Fit Models (Simulated)

In [ ]:
query = """
SELECT i.customer_id, i.invoice_no, ii.stock_code
FROM invoices i
JOIN invoice_items ii ON i.invoice_no = ii.invoice_no
WHERE i.is_cancellation = 0
LIMIT 10000
"""
df = query_to_dataframe(query)

# Quick fit for demonstration
pop = PopularityRecommender()
pop.fit(df)
hybrid = HybridRecommender()
hybrid.pop_rec = pop
# We would normally fit all component models here.

engine = ExplainabilityEngine(hybrid)

## 2. Product Affinity Network

In [ ]:
G = build_product_affinity_network(df, min_co_occurrence=15)
print(f"Network built with {G.number_of_nodes()} products and {G.number_of_edges()} co-purchases.")

communities = get_communities(G)
print(f"Detected {len(communities)} distinct product communities.")

# Plot largest connected component
if len(G.nodes) > 0:
    lcc = G.subgraph(max(nx.connected_components(G), key=len))
    plt.figure(figsize=(10,10))
    nx.draw_spring(lcc, node_size=20, node_color="blue", edge_color="gray", alpha=0.5)
    plt.title("Product Affinity Network (Largest Component)")
    plt.show()